# Coupled_Feedback — one-way coupling documentation

`MagneticCoupled` · ASN 28 validation form (TRUST-LiMe).



## Introduction

Documents the **one-way coupling** limitation of `force_lorentz`: the NS
velocity does **not** feed the magnetic convection.  With $\eta = 0$ and no
transport velocity, the magnetic field $\mathbf{b} = (y, -x)$ is an exact
equilibrium and must stay frozen — even when the NS velocity is non-zero.

## What is asserted

| # | Quantity | Formula |
|---|---|---|
| 1 | $\mathbf{b}$ stays frozen | $b_x(t) = y$, $b_y(t) = -x$ (machine precision) |
| 2 | hydrostatic balance holds | $\Delta p = -0.60$ (as in `MMS_Coupled`) |

The initial velocity $\mathbf{u}_0 = (0.1, 0)$ drives a transient flow that
decays; the magnetic field must **not** respond.  If the feedback were wired,
$\mathbf{b}$ would evolve.  This form is a **documentation test**: it pins the
current one-way behaviour so a future feedback implementation is noticed.



In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()



## Helpers: read a TRUST probe (`.son`) file


In [ ]:
import numpy as np
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, sonde):
    """Return (time, values) for probe `sonde` of case `case`."""
    path = f"{BUILD}/{case}_{sonde}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

print("BUILD =", BUILD)



## Run

Runs on **2 processors** (`nbProcs=2` + `case.partition()`).


In [ ]:
import pathlib, shutil

from trustutils import run

NB_PROCS = 2

run.reset()
case = run.addCase(".", "coupled_feedback.data", nbProcs=NB_PROCS)
case.partition()
run.printCases()
run.runCases()

build = pathlib.Path(run.BUILD_DIRECTORY)
build.mkdir(parents=True, exist_ok=True)
for son in pathlib.Path("src").glob(f"{case.dataFileName_}_*.son"):
    shutil.copy2(son, build / son.name)



## Checks

### #1 — magnetic field stays frozen (one-way coupling)


In [ ]:
t_bx, bx = read_son(case.dataFileName_, "SONDE_BX")
_, by = read_son(case.dataFileName_, "SONDE_BY")
# b = (y, -x): at the probe (0.35, 0.35), bx = 0.35, by = -0.35
bx_expected, by_expected = 0.35, -0.35
err_bx = float(np.max(np.abs(bx - bx_expected)))
err_by = float(np.max(np.abs(by - by_expected)))
print(f"max |bx - y|   = {err_bx:.3e}  (expected 0, machine precision)")
print(f"max |by - (-x)| = {err_by:.3e}  (expected 0, machine precision)")
assert err_bx < 1e-10, f"bx evolved: max |bx - y| = {err_bx}"
assert err_by < 1e-10, f"by evolved: max |by + x| = {err_by}"
print("OK: b stays frozen — the NS velocity does not feed the magnetic convection.")



### #2 — hydrostatic balance (as in MMS_Coupled)


In [ ]:
_, p1 = read_son(case.dataFileName_, "SONDE_P1")
_, p2 = read_son(case.dataFileName_, "SONDE_P2")
dp = p2[-1] - p1[-1]
expected = -(0.65**2 + 0.65**2 - 0.35**2 - 0.35**2)
err = abs(dp - expected) / abs(expected)
print(f"dp = {dp:.6g}   (analytic {expected:.4g}, err {err:.3e})")
assert dp < 0 and err < 0.05, f"pressure balance off: dp = {dp}"
print("OK: hydrostatic balance holds with non-zero initial velocity.")



## Computer Performance


In [ ]:
run.tablePerf()

